## Problem 2: CAPM vs Fama-French 3-Factor Model

**Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

**Read Fama-French 3-Factor data**

In [2]:
ff3_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"

ff3_raw = pd.read_csv(
    ff3_url,
    compression="zip",
    skiprows=3
)

ff3_raw.head()

,Unnamed: 0,Mkt-RF,SMB,HML,RF
0,192607,2.89,-2.55,-2.39,0.22
1,192608,2.64,-1.14,3.81,0.25
2,192609,0.38,-1.36,0.05,0.23
3,192610,-3.27,-0.14,0.82,0.32
4,192611,2.54,-0.11,-0.61,0.31


**Clean Fama-French factor data**

In [3]:
ff3 = ff3_raw.copy()

ff3 = ff3.rename(columns={ff3.columns[0]: "Date"})

ff3 = ff3[ff3["Date"].astype(str).str.isnumeric()]
ff3 = ff3[ff3["Date"].astype(str).str.len() == 6]

ff3["Date"] = pd.to_datetime(ff3["Date"].astype(str), format="%Y%m")

ff3["Mkt-RF"] = pd.to_numeric(ff3["Mkt-RF"], errors="coerce")
ff3["SMB"] = pd.to_numeric(ff3["SMB"], errors="coerce")
ff3["HML"] = pd.to_numeric(ff3["HML"], errors="coerce")
ff3["RF"] = pd.to_numeric(ff3["RF"], errors="coerce")

ff3.head()

,Date,Mkt-RF,SMB,HML,RF
0,1926-07-01,2.89,-2.55,-2.39,0.22
1,1926-08-01,2.64,-1.14,3.81,0.25
2,1926-09-01,0.38,-1.36,0.05,0.23
3,1926-10-01,-3.27,-0.14,0.82,0.32
4,1926-11-01,2.54,-0.11,-0.61,0.31


**Read 10 Industry Portfolio data**

In [4]:
industry_url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/10_Industry_Portfolios_CSV.zip"

industry_raw = pd.read_csv(
    industry_url,
    compression="zip",
    skiprows=11
)

industry_raw.head()

,Unnamed: 0,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other
0,192607,1.44,13.90,4.70,-1.14,2.90,0.83,0.12,1.85,7.04,2.14
1,192608,3.99,3.70,2.80,3.43,2.66,2.17,-0.72,4.17,-1.70,4.35
2,192609,1.15,4.98,1.17,-3.30,-0.39,2.42,0.21,0.69,2.05,0.31
3,192610,-1.24,-8.39,-3.65,-0.78,-4.58,-0.11,-2.29,-0.57,-3.27,-2.85
4,192611,5.21,-0.17,4.27,0.01,4.71,1.63,6.45,5.43,4.40,2.13


**Keep only the first monthly return table**

In [5]:
industry = industry_raw.copy()

industry = industry.rename(columns={industry.columns[0]: "Date"})

industry["Date_as_text"] = industry["Date"].astype(str)

first_non_monthly_row = industry[
    ~industry["Date_as_text"].str.isnumeric()
].index[0]

industry = industry.loc[:first_non_monthly_row - 1]

industry.head()

,Date,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other,Date_as_text
0,192607,1.44,13.90,4.70,-1.14,2.90,0.83,0.12,1.85,7.04,2.14,192607
1,192608,3.99,3.70,2.80,3.43,2.66,2.17,-0.72,4.17,-1.70,4.35,192608
2,192609,1.15,4.98,1.17,-3.30,-0.39,2.42,0.21,0.69,2.05,0.31,192609
3,192610,-1.24,-8.39,-3.65,-0.78,-4.58,-0.11,-2.29,-0.57,-3.27,-2.85,192610
4,192611,5.21,-0.17,4.27,0.01,4.71,1.63,6.45,5.43,4.40,2.13,192611


**Clean industry return columns**

In [6]:
industry = industry.drop(columns=["Date_as_text"])

industry["Date"] = pd.to_datetime(industry["Date"].astype(str), format="%Y%m")

industry["NoDur"] = pd.to_numeric(industry["NoDur"], errors="coerce")
industry["Durbl"] = pd.to_numeric(industry["Durbl"], errors="coerce")
industry["Manuf"] = pd.to_numeric(industry["Manuf"], errors="coerce")
industry["Enrgy"] = pd.to_numeric(industry["Enrgy"], errors="coerce")
industry["HiTec"] = pd.to_numeric(industry["HiTec"], errors="coerce")
industry["Telcm"] = pd.to_numeric(industry["Telcm"], errors="coerce")
industry["Shops"] = pd.to_numeric(industry["Shops"], errors="coerce")
industry["Hlth"] = pd.to_numeric(industry["Hlth"], errors="coerce")
industry["Utils"] = pd.to_numeric(industry["Utils"], errors="coerce")
industry["Other"] = pd.to_numeric(industry["Other"], errors="coerce")

industry.head()

,Date,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other
0,1926-07-01,1.44,13.90,4.70,-1.14,2.90,0.83,0.12,1.85,7.04,2.14
1,1926-08-01,3.99,3.70,2.80,3.43,2.66,2.17,-0.72,4.17,-1.70,4.35
2,1926-09-01,1.15,4.98,1.17,-3.30,-0.39,2.42,0.21,0.69,2.05,0.31
3,1926-10-01,-1.24,-8.39,-3.65,-0.78,-4.58,-0.11,-2.29,-0.57,-3.27,-2.85
4,1926-11-01,5.21,-0.17,4.27,0.01,4.71,1.63,6.45,5.43,4.40,2.13


In [7]:
ff3.shape, industry.shape

((1198, 5), (1198, 11))

**Merge both datasets**

In [8]:
data = pd.merge(ff3, industry, on="Date", how="inner")

data.head()

,Date,Mkt-RF,SMB,HML,RF,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other
0,1926-07-01,2.89,-2.55,-2.39,0.22,1.44,13.90,4.70,-1.14,2.90,0.83,0.12,1.85,7.04,2.14
1,1926-08-01,2.64,-1.14,3.81,0.25,3.99,3.70,2.80,3.43,2.66,2.17,-0.72,4.17,-1.70,4.35
2,1926-09-01,0.38,-1.36,0.05,0.23,1.15,4.98,1.17,-3.30,-0.39,2.42,0.21,0.69,2.05,0.31
3,1926-10-01,-3.27,-0.14,0.82,0.32,-1.24,-8.39,-3.65,-0.78,-4.58,-0.11,-2.29,-0.57,-3.27,-2.85
4,1926-11-01,2.54,-0.11,-0.61,0.31,5.21,-0.17,4.27,0.01,4.71,1.63,6.45,5.43,4.40,2.13


**Create Energy Excess Return**

In [9]:
data["Enrgy_Excess"] = data["Enrgy"] - data["RF"]

data[["Date", "Enrgy", "RF", "Enrgy_Excess", "Mkt-RF", "SMB", "HML"]].head()

,Date,Enrgy,RF,Enrgy_Excess,Mkt-RF,SMB,HML
0,1926-07-01,-1.14,0.22,-1.36,2.89,-2.55,-2.39
1,1926-08-01,3.43,0.25,3.18,2.64,-1.14,3.81
2,1926-09-01,-3.30,0.23,-3.53,0.38,-1.36,0.05
3,1926-10-01,-0.78,0.32,-1.10,-3.27,-0.14,0.82
4,1926-11-01,0.01,0.31,-0.30,2.54,-0.11,-0.61


**Run Model 1: CAPM**

In [10]:
y = data["Enrgy_Excess"]

X_capm = data[["Mkt-RF"]]
X_capm = sm.add_constant(X_capm)

capm_model = sm.OLS(y, X_capm).fit()

print(capm_model.summary())

                            OLS Regression Results                            
Dep. Variable:           Enrgy_Excess   R-squared:                       0.539
Model:                            OLS   Adj. R-squared:                  0.539
Method:                 Least Squares   F-statistic:                     1400.
Date:                Sun, 21 Jun 2026   Prob (F-statistic):          1.63e-203
Time:                        00:36:14   Log-Likelihood:                -3462.3
No. Observations:                1198   AIC:                             6929.
Df Residuals:                    1196   BIC:                             6939.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1553      0.127      1.223      0.2

**Run Model 2: Fama-French 3-Factor Model**

In [11]:
X_ff3 = data[["Mkt-RF", "SMB", "HML"]]
X_ff3 = sm.add_constant(X_ff3)

ff3_model = sm.OLS(y, X_ff3).fit()

print(ff3_model.summary())

                            OLS Regression Results                            
Dep. Variable:           Enrgy_Excess   R-squared:                       0.575
Model:                            OLS   Adj. R-squared:                  0.574
Method:                 Least Squares   F-statistic:                     538.1
Date:                Sun, 21 Jun 2026   Prob (F-statistic):          3.78e-221
Time:                        00:36:14   Log-Likelihood:                -3414.2
No. Observations:                1198   AIC:                             6836.
Df Residuals:                    1194   BIC:                             6857.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0787      0.122      0.643      0.5

**Comparison Table**

In [12]:
comparison = pd.DataFrame({
    "Model": ["CAPM", "Fama-French 3-Factor"],
    "Alpha": [capm_model.params["const"], ff3_model.params["const"]],
    "Market Beta": [capm_model.params["Mkt-RF"], ff3_model.params["Mkt-RF"]],
    "SMB Coefficient": [np.nan, ff3_model.params["SMB"]],
    "HML Coefficient": [np.nan, ff3_model.params["HML"]],
    "R-squared": [capm_model.rsquared, ff3_model.rsquared],
    "Adjusted R-squared": [capm_model.rsquared_adj, ff3_model.rsquared_adj],
    "AIC": [capm_model.aic, ff3_model.aic],
    "BIC": [capm_model.bic, ff3_model.bic]
})

comparison

,Model,Alpha,Market Beta,SMB Coefficient,HML Coefficient,R-squared,Adjusted R-squared,AIC,BIC
0,CAPM,0.155298,0.887844,NaN,NaN,0.539317,0.538932,6928.556027,6938.732845
1,Fama-French 3-Factor,0.078736,0.868238,-0.158578,0.329766,0.574822,0.573754,6836.471948,6856.825583


**Extract p-Values**

In [13]:
pvalues_comparison = pd.DataFrame({
    "Variable": ["Alpha", "Mkt-RF", "SMB", "HML"],
    "CAPM p-value": [
        capm_model.pvalues["const"],
        capm_model.pvalues["Mkt-RF"],
        np.nan,
        np.nan
    ],
    "Fama-French 3-Factor p-value": [
        ff3_model.pvalues["const"],
        ff3_model.pvalues["Mkt-RF"],
        ff3_model.pvalues["SMB"],
        ff3_model.pvalues["HML"]
    ]
})

pvalues_comparison

,Variable,CAPM p-value,Fama-French 3-Factor p-value
0,Alpha,2.215315e-01,5.201568e-01
1,Mkt-RF,1.627060e-203,5.114523e-188
2,SMB,NaN,1.020930e-04
3,HML,NaN,2.503349e-20


This will help us answer: Are SMB and HML statistically useful for explaining Energy industry excess returns?

**Comparison of Residuals**

In [14]:
data["CAPM_Predicted"] = capm_model.predict(X_capm)
data["FF3_Predicted"] = ff3_model.predict(X_ff3)

data["CAPM_Residuals"] = data["Enrgy_Excess"] - data["CAPM_Predicted"]
data["FF3_Residuals"] = data["Enrgy_Excess"] - data["FF3_Predicted"]

residual_comparison = pd.DataFrame({
    "Model": ["CAPM", "Fama-French 3-Factor"],
    "Residual Mean": [
        data["CAPM_Residuals"].mean(),
        data["FF3_Residuals"].mean()
    ],
    "Residual Standard Deviation": [
        data["CAPM_Residuals"].std(),
        data["FF3_Residuals"].std()
    ]
})

residual_comparison

,Model,Residual Mean,Residual Standard Deviation
0,CAPM,-3.469679e-16,4.355883
1,Fama-French 3-Factor,5.634521e-16,4.184660


### Residual Comparison

The residual standard deviation decreases from 4.3559 under the CAPM model to 4.1847 under the Fama-French 3-Factor model. This indicates that the Fama-French model makes smaller typical prediction errors than the single-factor CAPM model. Since both models have residual means close to zero, the main difference lies in the spread of the residuals rather than systematic overprediction or underprediction. This supports the earlier conclusion that the Fama-French 3-Factor model provides a better fit for Energy industry excess returns.

### Interpretation of Results

The Fama-French 3-Factor model improves the explanation of Energy industry excess returns compared with the single-factor CAPM model. The R-squared increases from 0.5393 under CAPM to 0.5748 under the Fama-French 3-Factor model. The adjusted R-squared also increases from 0.5389 to 0.5738, suggesting that the additional explanatory power is meaningful even after accounting for the extra variables.

The market beta remains positive and statistically significant. In the Fama-French model, the market beta is 0.8682, meaning that a 1 percentage point increase in market excess return is associated with an approximately 0.8682 percentage point increase in Energy industry excess return, holding SMB and HML constant.

The SMB coefficient is negative and statistically significant, suggesting that the Energy industry portfolio behaves more like large-cap stocks than small-cap stocks. The HML coefficient is positive and statistically significant, suggesting that the Energy industry portfolio has value-stock characteristics.

The alpha decreases from 0.1553 in the CAPM model to 0.0787 in the Fama-French model. However, the alpha remains statistically insignificant. Therefore, there is no strong evidence of abnormal excess returns after controlling for market, size, and value factors.

Overall, the Fama-French 3-Factor model provides a better fit than CAPM for explaining Energy industry excess returns.

## END